# Evidence RAG v3

This version uses the same models as v2 for embedding and inference, but includes the use of [LangChain](https://www.langchain.com) for building the embeddings and running the inference, and [Chroma](https://www.langchain.com/blog/langchain-chroma) as a retrievable vector database. It also uses paragraph-aware chunking to improvee the quality of retrieved chunks.

In [ ]:
# Install and import libraries

%pip install requests beautifulsoup4 pdfminer.six lxml sentence-transformers chromadb
%pip install selenium google-colab-selenium langchain langchain-text-splitters tqdm
%pip install langchain-community langchain-core langchain-google-genai langchain-classic

import requests
import re
import io
import os
import time
import pickle
import torch
import numpy as np
import google_colab_selenium as gs
from IPython.display import display, Markdown
from google.colab import drive
from bs4 import BeautifulSoup
from pdfminer.high_level import extract_text
from urllib.parse import urljoin, urlparse
from pathlib import Path
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.embeddings.base import Embeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.vectorstores import Chroma

In [ ]:
# Gemini API setup

from google.colab import userdata
api_key = userdata.get("GEMINI_API_KEY")
os.environ["GOOGLE_API_KEY"] = api_key

In [ ]:
# Load reviews and guidelines from storage

fetch_new_docs = True
time.sleep(1)
to_load = input("Would you like to load pre-saved reviews and guidelines? (y/n) ")
try:
    if to_load.lower()[0] == "y":
        drive.mount('/content/drive')
        with open('/content/drive/MyDrive/rag_documents.pkl', 'rb') as f:
            documents = pickle.load(f)
        with open('/content/drive/MyDrive/rag_papers.pkl', 'rb') as f:
            papers = pickle.load(f)
        print("\nLoaded", len(documents), "guidelines and", len(papers), "review articles")
        fetch_new_docs = False
except Exception as e:
    raise(e)

Would you like to load pre-saved reviews and guidelines? (y/n) y
Mounted at /content/drive

Loaded 880 guidelines and 1003 review articles


In [ ]:
# Load chunks and embeddings from storage

fetch_new_vecs = True
time.sleep(1)
to_load = input("Would you like to load pre-saved chunks and embeddings? (y/n) ")
try:
    if to_load.lower()[0] == "y":
        drive.mount('/content/drive')
        with open('/content/drive/MyDrive/rag_v3_corpus.pkl', 'rb') as f:
            corpus = pickle.load(f)
        print("\nLoaded", len(corpus), "chunks")
        fetch_new_vecs = False
except Exception as e:
    raise(e)

Would you like to load pre-saved chunks and embeddings? (y/n) n


In [ ]:
# Guideline sites and review journals

sites = [
    "https://www.rch.org.au/clinicalguide",
    "https://eyeandear.org.au/health-professionals/clinical-practice-guidelines/",
    "https://dermnetnz.org/topics",
    "https://sti.guidelines.org.au",
    "https://www.cdc.gov/yellow-book/hcp/contents/index.html",
    "https://www.thewomens.org.au/health-professionals/clinical-resources/clinical-guidelines-gps"
]

journals = [
    "Aust J Gen Pract",
    "Am Fam Physician",
    "Dtsch Arztebl Int",
    "Can Fam Physician",
    "CMAJ",
    "Aust Prescr",
    "Postgrad Med"
]

In [ ]:
# PubMed search terms

journal_query = " OR ".join([f'"{j}"[jour]' for j in journals])

query = """
Review[pt]
AND free full text[sb]
AND "last 5 years"[pdat]
AND English[lang]
AND
(
""" + journal_query + "\n)"

In [ ]:
# Crawl guideline websites

def detect_content_type(url):
    try:
        r = requests.get(url, stream=True, timeout=10)
        content_type = r.headers.get("Content-Type", "").lower()
        content_disp = r.headers.get("Content-Disposition", "").lower()
        if "pdf" in content_type:
            return "pdf"
        if "app.prompt.org.au" in url:
            return "pdf_link"
        if "attachment" in content_disp and "pdf" in content_disp:
            return "pdf"
        if "download" in url:
            return "pdf"
        return "html"
    except:
        r = requests.head(url, allow_redirects=True)
        content_type = r.headers.get("Content-Type", "")
        if "pdf" in content_type.lower():
            return "pdf"
        return "html"

def extract_html_text(url):
    r = requests.get(url, headers=headers)
    soup = BeautifulSoup(r.text, "lxml")
    if (use_driver or len(soup) < 3):
        driver.get(url)
        try:
            element = WebDriverWait(driver, 10).until(
                EC.none_of(
                    EC.title_contains("Checking your browser"),
                    EC.title_contains("Just a moment")
                )
            )
            time.sleep(1)
        except:
            pass
        soup = BeautifulSoup(driver.page_source, "lxml")
    title = soup.title.text.strip('\r\n\t ') if soup.title else ""
    if title[:12] == "RACGP - AJGP":
        title = soup.find("h1").text.strip('\r\n\t ') + " - AJGP"
    sections = []
    for heading in soup.find_all(["h1", "h2", "h3", "h4", "h5", "strong"]):
        heading_text = heading.get_text(strip=True)
        content_parts = []
        for sibling in [*heading.next_siblings, *heading.parent.next_siblings]:
            if getattr(sibling, "name", None) in ["h1", "h2", "h3", "h4", "h5", "strong"]:
                break
            if sibling.name in ["p", "li"]:
                content_parts.append(sibling.get_text(strip=True))
            # handle nested lists
            if sibling.name == "ul":
                for li in sibling.find_all("li"):
                    content_parts.append(li.get_text(strip=True))
        text = " ".join(content_parts)
        if len(text) > 20:  # filter tiny sections
            sections.append({
                "heading": heading_text,
                "text": text
            })
    return title, sections

def extract_pdf_text(url):
    r = requests.get(url, allow_redirects=True)
    file_like = io.BytesIO(r.content)
    text = extract_text(file_like)
    sections = []
    current = {"heading": "Document", "text": ""}
    for line in text.split("\n"):
        line = line.strip()
        # detect headings (simple heuristic)
        if line.isupper() and len(line) < 80:
            if current["text"]:
                sections.append(current)
            current = {"heading": line, "text": ""}
        else:
            current["text"] += " " + line
    if current["text"]:
        sections.append(current)
    title = sections[0]["heading"] if sections else ""
    return title, sections

def file_exists_in_folder(path):
    # Custom condition: returns True if the folder is not empty
    def _predicate(driver):
        files = os.listdir(path)
        valid_files = [f for f in files if f.endswith('.pdf')]
        return len(valid_files) > 0
    return _predicate

def extract_downloaded_pdf_text(url):
    driver.get(url)

    # Wait for download (adjust based on file size)
    try:
        WebDriverWait(driver, 10).until(file_exists_in_folder(download_path))
        print("Downloaded", os.listdir(download_path))
    except Exception as e:
        print("Download timed out or failed.")

    # Get all files in the directory with their full paths
    files = [os.path.join(download_path, f) for f in os.listdir(download_path)]
    # Filter to include only files (ignoring subdirectories)
    files = [f for f in files if os.path.isfile(f)]
    pdf_files = [f for f in files if ".pdf" in f]
    title = ""

    if pdf_files:
        # Find the file with the maximum modification time (most recent)
        latest_file = max(pdf_files, key=os.path.getmtime)
        title = Path(latest_file).stem

        # Extract and process text from file
        text = extract_text(latest_file)
        sections = []
        current = {"heading": "Document", "text": ""}
        for line in text.split("\n"):
            line = line.strip()
            # detect headings (simple heuristic)
            if line.isupper() and len(line) < 80:
                if current["text"]:
                    sections.append(current)
                current = {"heading": line, "text": ""}
            else:
                current["text"] += " " + line
        if current["text"]:
            sections.append(current)

        # Delete the file
        for file in files:
            os.remove(file)
        return title, sections

    else:
        print("Unable to locate file")
        return None

def discover_links(base_url):
    global use_driver
    use_driver = False
    try:
        r = requests.get(base_url, timeout=(10,10), headers=headers)
        soup = BeautifulSoup(r.text, "lxml")
    except:
        driver.get(base_url)
        soup = BeautifulSoup(driver.page_source, "lxml")
    if len(soup.find_all("a", href=True)) == 0:
        driver.get(base_url)
        soup = BeautifulSoup(driver.page_source, "lxml")
        use_driver = True
    base_domain = urlparse(base_url).netloc
    subs = [
        ("rch.org.au", "rch.org.au/clinicalguide/guideline_index"),
        ("dermnetnz.org", "dermnetnz.org/topics"),
        ("cdc.gov", "cdc.gov/yellow-book/hcp")
    ]
    for dom, sub in subs:
        if dom in base_domain:
            base_domain = sub
    links = []
    for a in soup.find_all("a", href=True):
        url = urljoin(base_url, a["href"])
        domain = urlparse(url).netloc
        if (
            (base_domain in url
               and not "eyeandear.org.au" in domain
               and not "thewomens.org.au" in domain)
            or "prompt.org.au" in domain
            or "/download/" in url
            or url.endswith(".pdf")
        ):
            links.append(url)
    return list(set(links))

def extract_document(url):
    doc_type = detect_content_type(url)
    try:
        if doc_type == "pdf":
            t, s = extract_pdf_text(url)
        elif doc_type == "pdf_link":
            t, s = extract_downloaded_pdf_text(url)
        else:
            t, s = extract_html_text(url)
        return {
            "url": url,
            "title": t,
            "sections": s
        }
    except Exception as e:
        print("Couldn't extract", url)
        print(e)
        return None

if fetch_new_docs:
    documents = []

    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
    use_driver = False

    download_path = "/content/temp_downloads"
    if not os.path.exists(download_path):
        os.makedirs(download_path)
    files = [os.path.join(download_path, f) for f in os.listdir(download_path)]
    files = [f for f in files if os.path.isfile(f)]
    for file in files:
        print("Deleting", file)
        os.remove(file)

    options = Options()
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")

    # Set Chrome preferences for automatic downloads
    prefs = {
        "download.default_directory": download_path,
        "download.prompt_for_download": False,
        "download.directory_upgrade": True,
        "plugins.always_open_pdf_externally": True  # Bypasses the PDF viewer
    }
    options.add_experimental_option("prefs", prefs)
    options.page_load_strategy = 'eager'

    driver = gs.Chrome(options=options)

    for site in sites:
        print("\nCrawling", site)
        links = discover_links(site)
        for link in links:
            print(len(documents), end=": ")
            print("Extracting", link)
            doc = extract_document(link)
            if doc:
                documents.append(doc)

    driver.quit()
    print("\nExtracted", len(documents), "documents in total")

In [ ]:
# Search PubMed

BASE = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"
max_results = 1000
max_extract = 1000

def search_pubmed(query, max_results=max_results):
    url = f"{BASE}/esearch.fcgi"
    params = {
        "db": "pubmed",
        "term": query,
        "retmax": max_results,
        "retmode": "json"
    }
    r = requests.get(url, params=params).json()
    results = r["esearchresult"]["idlist"]
    print(len(results), "results found in PubMed search")
    return results

def get_pmcid(pmid, retries=3):
    url = "https://www.ncbi.nlm.nih.gov/pmc/utils/idconv/v1.0/"
    params = {
        "ids": pmid,
        "format": "json"
    }
    for attempt in range(retries):
        try:
            r = requests.get(url, params=params, timeout=10)
            data = r.json()
            records = data.get("records", [])
            if records and "pmcid" in records[0]:
                return records[0]["pmcid"]
            return None
        except Exception as e:
            print(f" (get_pmcid attempt {attempt+1} failed: {e})", end="")
            time.sleep(2 ** attempt)  # exponential backoff: 1s, 2s, 4s
    return None

def download_pmc_xml(pmcid):
    url = f"https://www.ncbi.nlm.nih.gov/pmc/articles/{pmcid}/?format=xml"
    r = requests.get(url)
    return r.text

def extract_pmc_text(xml):
    soup = BeautifulSoup(xml, "lxml")
    title = soup.title.text.strip('\r\n\t ')
    sections = []
    for sec in soup.find_all("sec"):
        heading = sec.find("title")
        heading_text = heading.get_text() if heading else "Section"
        paragraphs = sec.find_all("p")
        text = " ".join(p.get_text() for p in paragraphs)
        if text.strip():
            sections.append({
                "heading": heading_text,
                "text": text
            })
    return title, sections

def get_article_url(pmid):
    params = {
        "db": "pubmed",
        "id": pmid,
        "retmode": "xml"
    }
    r = requests.get(BASE + "/efetch.fcgi", params=params)
    soup = BeautifulSoup(r.text, "xml")
    article_ids = soup.find_all("ArticleId")
    for aid in article_ids:
        if aid.get("IdType") == "doi":
            return "https://doi.org/" + aid.text
    return None

if fetch_new_docs:
    papers = []
    pmids = search_pubmed(query)
    driver = gs.Chrome(options=options)

    for pmid in pmids:
        pmcid = get_pmcid(pmid)
        print("Extracting PMID", pmid, end="")
        if pmcid:
            url = "https://pmc.ncbi.nlm.nih.gov/articles/" + pmcid
            title, sections = extract_html_text(url)
            if not sections:
                use_driver = True
                title, sections = extract_html_text(url)
                use_driver = False
            print(":", title)
        else:
            url = get_article_url(pmid)
            if url:
                title, sections = extract_html_text(url)
                if not sections:
                    use_driver = True
                    title, sections = extract_html_text(url)
                    use_driver = False
                print(":", title)
            else:
                text = ""
                title = ""
                sections = []
                print(" - no PMCID")
                print("")
        if (sections and title != "Error: DOI Not Found" and title != "Just a moment..."):
            papers.append({
                "pmid": pmid,
                "url": url,
                "title": title,
                "sections": sections
            })
        else:
            print("  Couldn't extract PMID", pmid)
        if len(papers) >= max_extract:
            break

    driver.quit()
    print("\nExtracted", len(papers), "papers in total")

In [ ]:
# Extract Cochrane Reviews

query_cr = """
Review[pt]
AND free full text[sb]
AND "last 5 years"[pdat]
AND (therapy[Majr] OR diagnosis[Majr])
AND "Cochrane Database Syst Rev"[jour]
"""

max_cr_extract = 500

def extract_cochrane_review(url):
    r = requests.get(url)
    soup = BeautifulSoup(r.text, "lxml")
    if use_driver:
        driver.get(url)
        try:
            element = WebDriverWait(driver, 10).until(
                EC.none_of(
                    EC.title_contains("Checking your browser"),
                    EC.title_contains("Just a moment")
                )
            )
            time.sleep(1)
        except:
            pass
        soup = BeautifulSoup(driver.page_source, "lxml")
    title_tag = soup.find("h1")
    title = title_tag.get_text(strip=True) if title_tag else ""
    pls_text = []
    # find headings that match Plain Language Summary
    for heading in soup.find_all(["h2", "h3"]):
        h_text = heading.get_text(strip=True).lower()
        if "plain language" in h_text:
            # collect content until next heading
            for sib in heading.next_siblings:
                if getattr(sib, "name", None) in ["h2", "h3"]:
                    break
                if sib.name == "p":
                    pls_text.append(sib.get_text(strip=True))
                if sib.name == "div":
                    pls_text.extend(
                        p.get_text(strip=True)
                        for p in sib.find_all("p")
                    )
            break  # only need first match
    text = " ".join(pls_text)
    sections = [{
        "heading": "Plain language summary",
        "text": text
    }]
    return title, sections

if fetch_new_docs:
    max_extract = len(papers) + max_cr_extract
    pmids = search_pubmed(query_cr)
    print("\nExtracting Cochrane Reviews\n")
    driver = gs.Chrome(options=options)

    for pmid in pmids:
        pmcid = get_pmcid(pmid)
        print("Extracting PMID", pmid, end="")
        if pmcid:
            url = "https://pmc.ncbi.nlm.nih.gov/articles/" + pmcid
            title, sections = extract_cochrane_review(url)
            if not sections["text"]:
                use_driver = True
                title, sections = extract_cochrane_review(url)
                use_driver = False
            print(":", title)
        else:
            print(" - no PMCID")
            continue
        if sections["text"]:
            papers.append({
                "pmid": pmid,
                "url": url,
                "title": title,
                "sections": sections
            })
        else:
            print("  Couldn't extract PMID", pmid)
        if len(papers) >= max_extract:
            break

    driver.quit()
    print("\nExtracted", len(papers), "papers in total")

In [ ]:
# Embedding setup

class MiniLMEmbeddings(Embeddings):

    def __init__(self):
        self.model = SentenceTransformer(
            "sentence-transformers/all-MiniLM-L6-v2",
            device = "cuda" if torch.cuda.is_available() else "cpu"
        )

    def embed_documents(self, texts):
        return self.model.encode(
            texts,
            normalize_embeddings=True
        ).tolist()

    def embed_query(self, text):
        return self.model.encode(
            text,
            normalize_embeddings=True
        ).tolist()

embeddings = MiniLMEmbeddings()

In [ ]:
# Chunking and embedding

def chunk(doc, chunk_size=1000, overlap=100):
    chunks = []
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=overlap,
        separators=[
            "\n\n",   # paragraph
            "\n",     # line
            ". ",     # sentence
            " ",      # word
            ""
        ]
    )
    for section in doc["sections"]:
        text = section["text"]
        split_texts = splitter.split_text(text)
        for t in split_texts:
            chunks.append({
                "text": t,
                "title": doc["title"],
                "heading": section["heading"],
                "source": doc["url"]
            })
    return chunks

if fetch_new_vecs:
    corpus = []

    # This accounts for a bug in the previous versions of this tool (which
    # did not throw an error until this version) in which `papers` was formatted
    # differently between Pubmed extraction and Cochrane extraction.
    for paper in papers:
        if isinstance(paper["sections"], dict):
            paper["sections"] = [paper["sections"]]

    for paper in papers:
        for c in chunk(paper):
            corpus.append(c)

    for page in documents:
        for c in chunk(page):
            corpus.append(c)

    print(len(corpus), "chunks obtained")

    texts = [c["text"] for c in corpus]
    metadatas = [{"title": c["title"], "heading": c["heading"], "source": c["source"]} for c in corpus]

    batch_size = 100
    vectorstore = None

    # tdqm is utilised to add a progress bar for embedding
    for i in tqdm(range(0, len(texts), batch_size)):
        batch_texts = texts[i:i+batch_size]
        batch_meta = metadatas[i:i+batch_size]
        if vectorstore is None:
            vectorstore = Chroma.from_texts(
                texts=batch_texts,
                metadatas=batch_meta,
                embedding=embeddings,
                persist_directory="./rag_db"
            )
        else:
            vectorstore.add_texts(texts=batch_texts, metadatas=batch_meta)

else:
    vectorstore = Chroma(
        persist_directory="./rag_db",
        embedding_function=embeddings
    )

61551 chunks obtained


100%|██████████| 616/616 [04:26<00:00,  2.31it/s]


In [ ]:
# Query functions

def ask(question, show_chunks=False):
    results = retriever.invoke(question)

    if len(results) == 0:
        print("\nNo relevant sources found")
        return None

    if show_chunks:
        for r in results:
            print(r.page_content)

    response = qa_chain.invoke({"input": question})
    display(Markdown(response["answer"]))

    print("\nSources:")
    sources = []
    for doc in response["context"]:
        t, s = doc.metadata["title"], doc.metadata["source"]
        if s not in sources:
            sources.append(s)
            print(s, end="")
            if t:
                print(":", t)
            else:
                print("")

In [ ]:
# Save chatbot data to Drive

if fetch_new_docs:
    to_save = input("Would you like to save the database documents to your Drive? (y/n) ")
    try:
        if to_save.lower()[0] == "y":
            drive.mount('/content/drive')
            with open('/content/drive/MyDrive/rag_documents.pkl', 'xb') as f:
                pickle.dump(documents, f)
            with open('/content/drive/MyDrive/rag_papers.pkl', 'xb') as f:
                pickle.dump(papers, f)
            print("\nSaved", len(documents), "guidelines and", len(papers), "review articles")
    except FileExistsError:
        print("\nError: The file already exists. Please manually delete or rename files if you still wish to save data.")
    except Exception as e:
        print(e)

if fetch_new_vecs:
    to_save = input("Would you like to save the chunked documents to your Drive? (y/n) ")
    try:
        if to_save.lower()[0] == "y":
            drive.mount('/content/drive')
            with open('/content/drive/MyDrive/rag_v3_corpus.pkl', 'xb') as f:
                pickle.dump(corpus, f)
            print("\nSaved", len(corpus), "chunks")
    except FileExistsError:
        print("\nError: The file already exists. Please manually delete or rename files if you still wish to save data.")
    except Exception as e:
        print(e)

Would you like to save the chunked documents to your Drive? (y/n) y
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Saved 61551 chunks


## Using Gemini 3.1 Flash Lite

In [ ]:
# Model setup

llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=0
)

In [ ]:
# Retriever

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 10}
)

system_prompt = (
    "Use the medical evidence below to answer. "
    "Construct an answer as a short sentence or list of points. Don't just quote. "
    "If you don't know the answer, say you don't know. "
    "Context: {context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

question_answer_chain = create_stuff_documents_chain(llm, prompt)
qa_chain = create_retrieval_chain(retriever, question_answer_chain)

In [ ]:
# User query

print("Answers given by this chatbot are AI-generated and MAY CONTAIN ERRORS.")
print("Please check the provided sources.")

while True:
    question = input("\n\nAsk a question: ")
    if question.lower() == "break":
        break
    ask(question)

Answers given by this chatbot are AI-generated and MAY CONTAIN ERRORS.
Please check the provided sources.


Ask a question: How do you treat febrile convulsions?


Management of febrile convulsions includes the following:

*   **Manage the underlying cause:** Focus on treating the source of the fever.
*   **Seizure management:** If the seizure lasts longer than 5 minutes, provide active seizure treatment.
*   **Complex presentations:** Perform targeted investigations (such as blood glucose, electrolytes, and ECG) for complex febrile seizures or atypical presentations to exclude provoked seizures or seizure mimickers.
*   **Clinical assessment:** Investigations should be performed if the child appears seriously unwell.
*   **Avoid unnecessary testing:** Routine blood tests and neuroimaging are not recommended for simple febrile seizures as they are low yield and expose the child to unnecessary risks.
*   **Medication:** Ongoing treatment with antiepileptic drugs is not routinely recommended.


Sources:
https://www.rch.org.au/clinicalguide/guideline_index/Febrile_seizure/: Clinical Practice Guidelines : Febrile seizure
https://www.cdc.gov/yellow-book/hcp/post-travel-evaluation/post-travel-evaluation-of-the-ill-traveler.html: Post-Travel Evaluation of the Ill Traveler | Yellow Book | CDC
https://dermnetnz.org/topics/roseola: Roseola (viral rash): Causes, Symptoms, and Treatment — DermNet
https://www.cdc.gov/yellow-book/hcp/travel-associated-infections-diseases/leptospirosis.html: Leptospirosis | Yellow Book | CDC


Ask a question: How do you treat pyloric stenosis?


Pyloric stenosis is treated through the following steps:

*   **Correction of metabolic status:** Before surgery, any dehydration, electrolyte imbalances, and acid-base abnormalities (specifically metabolic alkalosis) must be corrected.
*   **Surgical intervention:** The definitive treatment is a pyloromyotomy.
*   **Timing:** Surgery is typically delayed until serum bicarbonate levels have normalized to reduce the risk of post-operative apnea.


Sources:
https://www.rch.org.au/clinicalguide/guideline_index/Pyloric_stenosis/: Clinical Practice Guidelines : Pyloric stenosis
https://www.rch.org.au/clinicalguide/guideline_index/Radiology_Guidelines_Acute_indications/: Clinical Practice Guidelines : Radiology - Acute indications
https://www.rch.org.au/clinicalguide/guideline_index/Whooping_Cough_Pertussis/: Clinical Practice Guidelines : Whooping cough (pertussis)


Ask a question: What are the features of a myocardial infarction?


...


Sources:
https://pmc.ncbi.nlm.nih.gov/articles/PMC9906028: Compression and Expansion of Morbidity: Secular Trends Among Cohorts of the Same Age - PMC
https://pmc.ncbi.nlm.nih.gov/articles/PMC9081942: Troponins in myocardial infarction and injury - PMC
https://pmc.ncbi.nlm.nih.gov/articles/PMC8671020: Diagnosis and management of acute coronary syndromes - PMC
https://www.rch.org.au/clinicalguide/guideline_index/Chest_pain/: Clinical Practice Guidelines : Chest pain


Ask a question: What causes an earache?


Earaches can be caused by conditions within the ear itself or referred pain from other areas:

*   **Ear-related causes:** Infections of the external ear canal (otitis externa) caused by bacteria or fungi, often triggered by moisture, trauma, or skin barrier breakdown.
*   **Dental pathology:** This is the most common cause of non-ear-related (nonotologic) ear pain, occurring because the ear and teeth share the same nerve pathway (CN V). Specific issues include pulpitis, apical periodontitis, and pericoronitis.
*   **Upper airway issues:** Infections like tonsillitis or pharyngitis, as well as laryngopharyngeal reflux (LPR), can cause referred ear pain through shared neural pathways.
*   **Oral mucosal lesions:** Benign or malignant lesions in the oral cavity, such as aphthous ulcers, can manifest as ear pain due to shared innervation.
*   **Allergic reactions:** Contact dermatitis of the external auditory canal can cause sudden swelling, itching, and pain.


Sources:
https://pmc.ncbi.nlm.nih.gov/articles/PMC10645450: Referred otalgia: Common causes and evidence-based strategies for assessment and management - PMC
https://dermnetnz.org/topics/otitis-externa: Otitis externa


Ask a question: How do you recognise a melanoma?


Melanoma can be recognized through the following methods:

*   **Clinical Rules:** Use the **ABCDE** rule (Asymmetry, Border irregularity, Colour variation, Diameter, Evolving) for superficial melanomas, and the **EFG** signs (Elevated, Firm to touch, Growing) for nodular melanomas.
*   **Visual Signs:** Look for the "ugly duckling" sign (a lesion that looks different from others on the body) and any freckle or mole that changes in size, shape, color, or elevation.
*   **Professional Assessment:** Dermatologists use dermoscopy to identify specific features like atypical pigment networks, blue-white veils, and irregular vascular patterns.
*   **Diagnostic Tools:** Clinical evaluation may be supported by total body photography, confocal microscopy, or genomic analysis.
*   **Definitive Diagnosis:** A biopsy and histopathological examination are required to confirm a diagnosis of melanoma.


Sources:
https://dermnetnz.org/topics/melanoma: Melanoma Skin Cancer: Images, Diagnosis, and Treatment — DermNet
https://www.cdc.gov/yellow-book/hcp/environmental-hazards-risks/sun-exposure-in-travelers.html: Sun Exposure in Travelers | Yellow Book | CDC
https://dermnetnz.org/topics/melasma: Melasma (facial pigmentation)
https://dermnetnz.org/topics/melanonychia: Melanonychia
https://dermnetnz.org/topics/lentigo-maligna-and-lentigo-maligna-melanoma: Lentigo maligna and lentigo maligna melanoma


Ask a question: break
